# Pipeline de Alfabetização no Brasil — Notebook 2: Silver Layer

**Tech Challenge Fase 2 — FIAP POSTECH**

---

## Objetivo

Neste notebook realizamos as transformações da camada **Silver**:

1. Limpeza de dados (tratamento de nulos, remoção de duplicatas)
2. Padronização de nomes de colunas e tipos de dados
3. Validação de consistência
4. **Integração das bases** (join entre indicadores, metas e dimensões)
5. Particionamento por `ano` + `sigla_uf` para otimizar queries Athena

## Fluxo

```
Bronze (Parquet bruto)  ──►  [limpeza + tipagem + integração]  ──►  Silver (Parquet particionado)
```

## 1. Imports e Configuração

In [ ]:
import os
import io
import logging
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import boto3

try:
    from dotenv import load_dotenv
    # Caminho absoluto para garantir que o .env seja encontrado
    # independente de onde o VS Code executa o kernel
    _env_path = Path(__file__).parent.parent / ".env" if "__file__" in dir() else Path.cwd().parent / ".env"
    _env_candidates = [
        Path.home() / "Desktop/tech-challenge-fase2/.env",
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
    ]
    for _p in _env_candidates:
        if _p.exists():
            load_dotenv(_p)
            break
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

S3_BUCKET    = os.getenv("S3_BUCKET_NAME",       "tech-challenge-alfabetizacao-01")
AWS_REGION   = os.getenv("AWS_DEFAULT_REGION",   "us-east-1")
USE_AWS      = os.getenv("USE_AWS", "false").lower() == "true"
LOCAL_BRONZE = Path.home() / "Desktop/tech-challenge-fase2/data/bronze"
LOCAL_SILVER = Path.home() / "Desktop/tech-challenge-fase2/data/silver"
LOCAL_SILVER.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
print(f"Modo : {'AWS S3' if USE_AWS else 'LOCAL'}")
print(f"Bucket: {S3_BUCKET}")
print(f"USE_AWS env: {os.getenv('USE_AWS')}")

## 2. Leitura da Camada Bronze

In [ ]:
s3_client = boto3.client("s3", region_name=AWS_REGION) if USE_AWS else None


def read_bronze_local(name: str) -> pd.DataFrame:
    """Lê Parquet da camada Bronze local."""
    path = LOCAL_BRONZE / f"{name}.parquet"
    if not path.exists():
        logger.error("Arquivo Bronze não encontrado: %s. Execute notebook 01 primeiro.", path)
        return pd.DataFrame()
    df = pd.read_parquet(path)
    # Remove metadados de ingestão para processamento
    meta_cols = [c for c in df.columns if c.startswith("_")]
    return df.drop(columns=meta_cols)


def read_bronze_s3(name: str) -> pd.DataFrame:
    """Lê Parquet mais recente do S3 Bronze."""
    prefix = f"bronze/{name}/"
    resp = s3_client.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix)
    objects = sorted(
        [o for o in resp.get("Contents", []) if o["Key"].endswith(".parquet")],
        key=lambda o: o["LastModified"], reverse=True
    )
    if not objects:
        logger.error("Nenhum arquivo Bronze em s3://%s/%s", S3_BUCKET, prefix)
        return pd.DataFrame()
    obj = s3_client.get_object(Bucket=S3_BUCKET, Key=objects[0]["Key"])
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    meta_cols = [c for c in df.columns if c.startswith("_")]
    return df.drop(columns=meta_cols)


def read_bronze(name: str) -> pd.DataFrame:
    return read_bronze_s3(name) if USE_AWS else read_bronze_local(name)


# Carrega todas as tabelas Bronze
meta_brasil     = read_bronze("meta_brasil")
meta_uf         = read_bronze("meta_uf")
meta_municipio  = read_bronze("meta_municipio")
indicador_uf    = read_bronze("indicador_uf")
indicador_mun   = read_bronze("indicador_municipio")

print("Bronze carregado:")
for name, df in [("meta_brasil",meta_brasil),("meta_uf",meta_uf),
                  ("meta_municipio",meta_municipio),("indicador_uf",indicador_uf),
                  ("indicador_municipio",indicador_mun)]:
    print(f"  {name:30s}: {len(df):>7,} linhas | colunas: {df.columns.tolist()}")

## 3. Transformações Silver

### 3.1 Limpeza Geral

In [ ]:
META_COLS_2030 = [
    "meta_alfabetizacao_2024", "meta_alfabetizacao_2025",
    "meta_alfabetizacao_2026", "meta_alfabetizacao_2027",
    "meta_alfabetizacao_2028", "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030"
]


def padroniza_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase, strip, substitui espaços por underscore."""
    df = df.copy()
    df.columns = [c.lower().strip().replace(" ", "_") for c in df.columns]
    return df


def converte_numericos(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Converte colunas para numérico, coerce erros → NaN."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def clean_meta_brasil(df: pd.DataFrame) -> pd.DataFrame:
    df = padroniza_colunas(df)
    df = converte_numericos(df, ["ano", "taxa_alfabetizacao", "percentual_participacao"] + META_COLS_2030)
    df["ano"] = df["ano"].astype("Int64")
    df = df.drop_duplicates(subset=["ano", "rede"])
    df["rede"] = df["rede"].str.strip().str.title()
    logger.info("meta_brasil silver: %d linhas", len(df))
    return df


def clean_meta_uf(df: pd.DataFrame) -> pd.DataFrame:
    df = padroniza_colunas(df)
    df = converte_numericos(df, ["ano", "taxa_alfabetizacao", "percentual_participacao"] + META_COLS_2030)
    df["ano"] = df["ano"].astype("Int64")
    df["sigla_uf"] = df["sigla_uf"].str.strip().str.upper()
    df = df.drop_duplicates(subset=["ano", "sigla_uf", "rede"])
    df = df.dropna(subset=["sigla_uf"])
    logger.info("meta_uf silver: %d linhas", len(df))
    return df


def clean_meta_municipio(df: pd.DataFrame) -> pd.DataFrame:
    df = padroniza_colunas(df)
    df = converte_numericos(df, ["ano", "taxa_alfabetizacao", "percentual_participacao"] + META_COLS_2030)
    df["ano"] = df["ano"].astype("Int64")
    # Garante 7 dígitos no código do município
    df["id_municipio"] = df["id_municipio"].astype(str).str.zfill(7)
    df = df.drop_duplicates(subset=["ano", "id_municipio", "rede"])
    df = df.dropna(subset=["id_municipio", "ano"])
    logger.info("meta_municipio silver: %d linhas", len(df))
    return df


def clean_indicador_uf(df: pd.DataFrame) -> pd.DataFrame:
    df = padroniza_colunas(df)
    df = converte_numericos(df, [
        "ano", "taxa_alfabetizacao", "media_portugues",
        "proporcao_aluno_nivel_0", "proporcao_aluno_nivel_1", "proporcao_aluno_nivel_2",
        "proporcao_aluno_nivel_3", "proporcao_aluno_nivel_4", "proporcao_aluno_nivel_5"
    ])
    df["ano"] = df["ano"].astype("Int64")
    df["sigla_uf"] = df["sigla_uf"].str.strip().str.upper()
    df["serie"] = pd.to_numeric(df["serie"], errors="coerce").astype("Int64")
    df["rede"] = pd.to_numeric(df["rede"], errors="coerce").astype("Int64")
    # Filtra taxa válida (0–100)
    mask = (df["taxa_alfabetizacao"].isna() |
            df["taxa_alfabetizacao"].between(0, 100))
    df = df[mask]
    df = df.drop_duplicates(subset=["ano", "sigla_uf", "serie", "rede"])
    df = df.dropna(subset=["sigla_uf", "ano"])
    logger.info("indicador_uf silver: %d linhas", len(df))
    return df


def clean_indicador_municipio(df: pd.DataFrame) -> pd.DataFrame:
    df = padroniza_colunas(df)
    df = converte_numericos(df, [
        "ano", "taxa_alfabetizacao", "media_portugues",
        "proporcao_aluno_nivel_0", "proporcao_aluno_nivel_1", "proporcao_aluno_nivel_2",
        "proporcao_aluno_nivel_3", "proporcao_aluno_nivel_4", "proporcao_aluno_nivel_5"
    ])
    df["ano"] = df["ano"].astype("Int64")
    df["id_municipio"] = df["id_municipio"].astype(str).str.zfill(7)
    df["serie"] = pd.to_numeric(df["serie"], errors="coerce").astype("Int64")
    df["rede"] = pd.to_numeric(df["rede"], errors="coerce").astype("Int64")
    mask = (df["taxa_alfabetizacao"].isna() |
            df["taxa_alfabetizacao"].between(0, 100))
    df = df[mask]
    df = df.drop_duplicates(subset=["ano", "id_municipio", "serie", "rede"])
    df = df.dropna(subset=["id_municipio", "ano"])
    logger.info("indicador_municipio silver: %d linhas", len(df))
    return df


# Executa limpezas
s_meta_brasil    = clean_meta_brasil(meta_brasil)
s_meta_uf        = clean_meta_uf(meta_uf)
s_meta_municipio = clean_meta_municipio(meta_municipio)
s_ind_uf         = clean_indicador_uf(indicador_uf)
s_ind_mun        = clean_indicador_municipio(indicador_mun)

print("Limpeza Silver concluída.")

### 3.2 Integração das Bases (Silver Unificada — Município)

In [ ]:
META_COLS = [
    "meta_alfabetizacao_2024", "meta_alfabetizacao_2025", "meta_alfabetizacao_2026",
    "meta_alfabetizacao_2027", "meta_alfabetizacao_2028", "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030"
]

UF_MAP = {
    "11":"RO","12":"AC","13":"AM","14":"RR","15":"PA","16":"AP","17":"TO",
    "21":"MA","22":"PI","23":"CE","24":"RN","25":"PB","26":"PE","27":"AL",
    "28":"SE","29":"BA","31":"MG","32":"ES","33":"RJ","35":"SP","41":"PR",
    "42":"SC","43":"RS","50":"MS","51":"MT","52":"GO","53":"DF"
}


def seleciona_colunas(df, colunas):
    """Retorna apenas as colunas que existem no DataFrame."""
    existentes = []
    for c in colunas:
        if c in df.columns:
            existentes.append(c)
    return df[existentes]


def integra_municipio(ind_mun, meta_mun, meta_uf_df, meta_brasil_df):
    df = ind_mun.copy()
    df["sigla_uf"] = df["id_municipio"].str[:2].map(UF_MAP)

    # Metas municipais — join por id_municipio + ano
    colunas_mun = ["id_municipio", "ano"] + META_COLS
    meta_mun_slim = seleciona_colunas(meta_mun, colunas_mun)
    meta_mun_slim = meta_mun_slim.drop_duplicates(subset=["id_municipio", "ano"])
    meta_mun_slim = meta_mun_slim.rename(columns={"meta_alfabetizacao_2030": "meta_mun_2030"})
    df = df.merge(meta_mun_slim, on=["id_municipio", "ano"], how="left", suffixes=("", "_meta"))

    # Metas estaduais — join por sigla_uf + ano
    colunas_uf = ["sigla_uf", "ano", "meta_alfabetizacao_2030"]
    meta_uf_slim = seleciona_colunas(meta_uf_df, colunas_uf)
    meta_uf_slim = meta_uf_slim.drop_duplicates(subset=["sigla_uf", "ano"])
    meta_uf_slim = meta_uf_slim.rename(columns={"meta_alfabetizacao_2030": "meta_uf_2030"})
    df = df.merge(meta_uf_slim, on=["sigla_uf", "ano"], how="left")

    # Meta nacional — join por ano
    colunas_br = ["ano", "meta_alfabetizacao_2030"]
    meta_br_slim = seleciona_colunas(meta_brasil_df, colunas_br)
    meta_br_slim = meta_br_slim.drop_duplicates(subset=["ano"])
    meta_br_slim = meta_br_slim.rename(columns={"meta_alfabetizacao_2030": "meta_brasil_2030"})
    df = df.merge(meta_br_slim, on=["ano"], how="left")

    # Gaps em relação às metas
    if "taxa_alfabetizacao" in df.columns and "meta_mun_2030" in df.columns:
        df["gap_meta_municipio_2030"] = (df["taxa_alfabetizacao"] - df["meta_mun_2030"]).round(2)
    if "taxa_alfabetizacao" in df.columns and "meta_uf_2030" in df.columns:
        df["gap_meta_uf_2030"] = (df["taxa_alfabetizacao"] - df["meta_uf_2030"]).round(2)
        df["atingiu_meta_uf"]  = df["gap_meta_uf_2030"] >= 0

    df["_data_processamento"] = RUN_TS
    return df


silver_municipio = integra_municipio(s_ind_mun, s_meta_municipio, s_meta_uf, s_meta_brasil)

print(f"Silver município integrada: {len(silver_municipio):,} linhas")
print(f"Colunas: {silver_municipio.columns.tolist()}")
display(silver_municipio.head(5))

### 3.3 Integração UF (Silver UF)

In [ ]:
def integra_uf(ind_uf, meta_uf_df, meta_brasil_df):
    df = ind_uf.copy()

    # Metas estaduais
    colunas_uf = ["sigla_uf", "ano", "meta_alfabetizacao_2030"]
    meta_uf_slim = seleciona_colunas(meta_uf_df, colunas_uf)
    meta_uf_slim = meta_uf_slim.drop_duplicates(subset=["sigla_uf", "ano"])
    meta_uf_slim = meta_uf_slim.rename(columns={"meta_alfabetizacao_2030": "meta_uf_2030"})
    df = df.merge(meta_uf_slim, on=["sigla_uf", "ano"], how="left")

    # Meta nacional
    colunas_br = ["ano", "meta_alfabetizacao_2030"]
    meta_br_slim = seleciona_colunas(meta_brasil_df, colunas_br)
    meta_br_slim = meta_br_slim.drop_duplicates(subset=["ano"])
    meta_br_slim = meta_br_slim.rename(columns={"meta_alfabetizacao_2030": "meta_brasil_2030"})
    df = df.merge(meta_br_slim, on=["ano"], how="left")

    if "taxa_alfabetizacao" in df.columns and "meta_uf_2030" in df.columns:
        df["gap_meta_uf_2030"] = (df["taxa_alfabetizacao"] - df["meta_uf_2030"]).round(2)
        df["atingiu_meta_uf"]  = df["gap_meta_uf_2030"] >= 0

    df["_data_processamento"] = RUN_TS
    return df


silver_uf = integra_uf(s_ind_uf, s_meta_uf, s_meta_brasil)

print(f"Silver UF integrada: {len(silver_uf):,} linhas")
display(silver_uf.head(5))

## 4. Persistência da Camada Silver

In [ ]:
def save_silver(df, name, partition_cols=None):
    """Salva Silver localmente (particionado) e no S3 (arquivo único por tabela)."""
    base = LOCAL_SILVER / name
    base.mkdir(parents=True, exist_ok=True)

    if partition_cols and all(c in df.columns for c in partition_cols):
        for keys, grupo in df.groupby(partition_cols):
            if not isinstance(keys, tuple):
                keys = (keys,)
            subdir = base
            for col, val in zip(partition_cols, keys):
                subdir = subdir / f"{col}={val}"
            subdir.mkdir(parents=True, exist_ok=True)
            grupo.drop(columns=list(partition_cols), errors="ignore").to_parquet(
                subdir / "data.parquet", index=False, engine="pyarrow"
            )
    else:
        df.to_parquet(base / "data.parquet", index=False, engine="pyarrow")

    print(f"  Local silver/{name}: {len(df):,} linhas")

    if USE_AWS:
        s3_client = boto3.client("s3", region_name=AWS_REGION)
        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)
        key = f"silver/{name}/{name}.parquet"
        s3_client.put_object(Bucket=S3_BUCKET, Key=key, Body=buffer.getvalue())
        print(f"  S3    silver/{name}: s3://{S3_BUCKET}/{key}")


save_silver(s_meta_brasil,    "meta_brasil")
save_silver(s_meta_uf,        "meta_uf")
save_silver(s_meta_municipio, "meta_municipio")
save_silver(s_ind_uf,         "indicador_uf",           partition_cols=["sigla_uf", "ano"])
save_silver(silver_municipio, "alfabetizacao_municipio", partition_cols=["sigla_uf", "ano"])

print(f"\nSilver concluído. Modo: {'AWS S3' if USE_AWS else 'LOCAL'}")

## 5. Estatísticas da Silver

In [ ]:
print("=" * 60)
print("Estatísticas Silver — Indicador Município")
print("=" * 60)

df_stats = silver_municipio.copy()
print(f"\nAnos disponíveis: {sorted(df_stats['ano'].dropna().astype(int).unique())}")
print(f"Municípios únicos: {df_stats['id_municipio'].nunique():,}")
print(f"UFs cobertas: {df_stats['sigla_uf'].nunique()}")

if "taxa_alfabetizacao" in df_stats.columns:
    print("\nEstatísticas da Taxa de Alfabetização:")
    display(df_stats.groupby("ano")["taxa_alfabetizacao"].describe().round(2))

In [ ]:
print("Municípios que atingiram meta estadual 2030, por UF (2023):")
if "atingiu_meta_uf" in silver_municipio.columns:
    ano_max = silver_municipio["ano"].max()
    resumo = (
        silver_municipio[silver_municipio["ano"] == ano_max]
        .groupby("sigla_uf")["atingiu_meta_uf"]
        .agg(total="count", atingiram="sum")
        .assign(pct=lambda x: (x["atingiram"] / x["total"] * 100).round(1))
        .sort_values("pct", ascending=False)
    )
    display(resumo.head(10))
else:
    print("Coluna 'atingiu_meta_uf' não disponível — verifique o join Silver.")

---
## Decisões Arquiteturais — Silver Layer

| Decisão | Escolha | Motivo |
|---|---|---|
| Integração na Silver | Join de 5 tabelas | Reduz complexidade da Gold; cada join é auditável isoladamente |
| Particionamento `sigla_uf / ano` | Colunas de filtro frequente | Athena só lê partições relevantes → reduz custo por query em até 90% |
| `id_municipio` com `zfill(7)` | Normalização de chave | Código IBGE pode vir com ou sem zeros à esquerda |
| Manter `rede` nas chaves de join | Escola pública ≠ privada | Indicadores divergem significativamente entre redes |
| Calculação de `gap_meta_*` na Silver | Facilita agregações Gold | Evita recalcular em cada query downstream |

**Próximo passo:** execute `03_gold_analytics.ipynb`